# Day 14 — CAC, ARPC and LTV assessment

This notebook reads the processed Week 3 customer-month revenue and LTV input files. Revenue is in unconfirmed source amount units. Marketing spend is not present in the supplied source tables, so CAC is not calculable.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
ROOT = Path.cwd()
DATA = ROOT / 'data/processed/week3'
revenue = pd.read_csv(DATA / 'customer_month_revenue.csv', parse_dates=['activity_month'])
ltv = pd.read_csv(DATA / 'customer_revenue_ltv_inputs.csv', parse_dates=['first_active_month','last_active_month'])
revenue.head()

## Average revenue per active paid customer (ARPC)

For each month: total customer-level recurring MRR divided by distinct active paid accounts in that month. This is an active-customer monthly ARPC, not revenue per all registered accounts.

In [ ]:
arpc = (revenue.groupby('activity_month').agg(total_mrr_source_amount=('customer_month_mrr','sum'), active_paid_customers=('account_id','nunique')).reset_index())
arpc['arpc_source_amount_per_active_customer'] = arpc['total_mrr_source_amount'] / arpc['active_paid_customers'].replace(0, pd.NA)
arpc

In [ ]:
ax = arpc.plot(x='activity_month', y='arpc_source_amount_per_active_customer', marker='o', title='Monthly ARPC (source amount units)')
ax.set_ylabel('Source amount units / active paid customer')
plt.tight_layout()
plt.show()

## CAC feasibility

The supplied accounts/subscriptions/churn/feature/support data do not include campaign or channel-level marketing expenditure. `referral_source` is an acquisition descriptor, not spend. Therefore CAC = acquisition spend / new customers cannot be calculated. Do not substitute zero spend.

## LTV feasibility decision

The dataset provides a finite observed history and account-level revenue linkage, but does not establish a complete customer lifetime, reliable customer-level churn timing, or a validated forward churn/survival assumption. The notebook therefore preserves observed revenue and tenure inputs but deliberately does not publish a realized or forecast LTV. A future model requires an approved churn definition, sufficient follow-up, and a documented horizon/discounting policy.

In [ ]:
ltv[['account_id','observed_revenue_source_amount','observed_active_months','tenure_months_observed','ltv_status']].head()